# NumPy: Arrays, Vectorisation and Numerical Computing

Notebook for **Chapter 16** of the *python* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/python/ch16.html

Tip: change a number, predict the output, then run the cell again.

## 16.1 What NumPy Is, When to Use It, and the Mental Model

In [ ]:
import numpy as np
print("major version:", np.__version__.split(".")[0])
values = [1.5, 2.0, 3.5, 4.0]
as_list = sum(v * v for v in values)
arr = np.array(values)
as_array = (arr * arr).sum()
print(as_list, as_array, as_list == as_array)
print(type(arr), arr.dtype, arr.shape)

major version: 2
34.5 34.5 True
<class 'numpy.ndarray'> float64 (4,)


## 16.2 The ndarray: Attributes, dtypes and Memory Layout

Before you create arrays in bulk, learn to read one. These attributes answer most "why is my array behaving oddly" questions.

In [ ]:
import numpy as np
a = np.zeros((1000, 1000))
print(a.shape, a.ndim, a.size, a.dtype)
print(a.itemsize, a.nbytes, a.strides)
b = a.astype(np.float32)
c = a.astype(np.uint8)
print(a.nbytes, b.nbytes, c.nbytes)
print(np.array([1, 2, 3]).dtype, np.array([1, 2.5]).dtype, np.array([1, "a"]).dtype)

(1000, 1000) 2 1000000 float64
8 8000000 (8000, 8)
8000000 4000000 1000000
int64 float64 <U21


In [ ]:
import numpy as np
a = np.arange(6, dtype=np.int32).reshape(2, 3)
t = a.T
print(a.strides, t.strides)
print(a.flags["C_CONTIGUOUS"], t.flags["C_CONTIGUOUS"], t.flags["F_CONTIGUOUS"])
print(t.base is a, np.shares_memory(a, t))
c = np.ascontiguousarray(t)
print(c.strides, np.shares_memory(a, c))
print(t.ravel(), t.ravel(order="K"))

(12, 4) (4, 12)
True False True
False True
(8, 4) False
[0 3 1 4 2 5] [0 1 2 3 4 5]


## 16.3 Creating Arrays

There are four ways arrays are born: from existing Python data, from a pattern (zeros, ranges), from random numbers (16.10), and from files. Pick the constructor that says what you mean.

In [ ]:
import numpy as np
print(np.arange(0, 10, 3))
print(np.arange(0, 1, 0.25))
print(np.linspace(0, 1, 5))
print(np.full((2, 3), 7))
print(np.eye(3, dtype=int))
x = np.array([1.5, 2.5])
print(np.zeros_like(x), np.zeros_like(x).dtype)
print(np.logspace(0, 3, 4))

[0 3 6 9]
[0.   0.25 0.5  0.75]
[0.   0.25 0.5  0.75 1.  ]
[[7 7 7]
 [7 7 7]]
[[1 0 0]
 [0 1 0]
 [0 0 1]]
[0. 0.] float64
[   1.   10.  100. 1000.]


## 16.4 Indexing: Slicing, Boolean Masks and Fancy Indexing

Indexing is how you say "give me these elements". NumPy has three families and they behave differently in one crucial way: slices are **views** (no copy), while boolean and fancy indexing return **copies**.

In [ ]:
import numpy as np
sales = np.array([120, 340, 95, 410, 80, 275, 1500, 310])
mask = sales > 300
print(mask)
print(sales[mask])
print(sales[(sales > 100) & (sales < 400)])
print(sales[~mask])
print(mask.sum(), mask.mean())
capped = sales.copy()
capped[capped > 500] = 500
print(capped)
print(sales[sales > 300].mean())

[False  True False  True False False  True  True]
[ 340  410 1500  310]
[120 340 275 310]
[120  95  80 275]
4 0.5
[120 340  95 410  80 275 500 310]
640.0


In [ ]:
import numpy as np
m = np.arange(20).reshape(4, 5)
print(m[[3, 0]])
print(m[[0, 2], [1, 4]])
print(m[np.ix_([0, 2], [1, 4])])
sub = m[[0, 1]]
sub[0, 0] = 999
print(m[0, 0], sub.base is None)
view = m[:2]
view[0, 0] = 999
print(m[0, 0])

[[15 16 17 18 19]
 [ 0  1  2  3  4]]
[ 1 14]
[[ 1  4]
 [11 14]]
0 True
999


## 16.5 Broadcasting: Combining Arrays of Different Shapes

In [ ]:
# This cell is meant to fail: read the error message, that is the lesson.
import numpy as np
A = np.array([[0, 0, 0], [10, 10, 10], [20, 20, 20]])
b = np.array([0, 1, 2])
print(A + b)
col = np.array([[1], [2], [3]])
row = np.array([[10, 20, 30, 40]])
print((col * row).shape)
print(col * row)
try:
    np.ones(3) + np.ones(4)
except ValueError as e:
    print("ValueError:", e)
m = np.arange(6).reshape(2, 3)
w = np.array([100, 200])
try:
    m + w
except ValueError as e:
    print("ValueError:", e)
print(m + w[:, None])

[[ 0  1  2]
 [10 11 12]
 [20 21 22]]
(3, 4)
[[ 10  20  30  40]
 [ 20  40  60  80]
 [ 30  60  90 120]]
ValueError: operands could not be broadcast together with shapes (3,) (4,) 
ValueError: operands could not be broadcast together with shapes (2,3) (2,) 
[[100 101 102]
 [203 204 205]]


## 16.6 Ufuncs, Reductions and the axis Argument

A **ufunc** (universal function) is a function that applies the same operation to every element, in compiled code, with broadcasting built in. `+`, `*`, `np.sqrt` and `np.exp` are all ufuncs. A **reduction** collapses an axis into one number, such as a sum or maximum.

In [ ]:
import numpy as np
sales = np.arange(24).reshape(2, 3, 4)
print(sales.sum(axis=(1, 2)))
print(sales.sum(axis=(0, 1)))
print(sales.argmax(axis=2))
print(sales.mean(axis=1).shape, sales.mean(axis=1, keepdims=True).shape)
print(sales.cumsum(axis=1)[0])
print(np.diff(sales, axis=2)[0, 0])

[ 66 210]
[60 66 72 78]
[[3 3 3]
 [3 3 3]]
(2, 4) (2, 1, 4)
[[ 0  1  2  3]
 [ 4  6  8 10]
 [12 15 18 21]]
[1 1 1]


## 16.7 Reshape, Transpose, Stack and Split

Real data rarely arrives in the shape a function wants. These tools rearrange arrays; most of them change only the header, so they are cheap.

In [ ]:
import numpy as np
a = np.array([[1, 2], [3, 4]])
b = np.array([[5, 6], [7, 8]])
print(np.concatenate([a, b], axis=0).shape)
print(np.concatenate([a, b], axis=1).shape)
print(np.stack([a, b]).shape, np.stack([a, b], axis=2).shape)
print(np.vstack([a, b]).shape, np.hstack([a, b]).shape)
x = np.array([1, 2, 3])
y = np.array([10, 20, 30])
print(np.column_stack([x, y]))
parts = np.array_split(np.arange(10), 3)
print([p.tolist() for p in parts])
print(np.roll(np.arange(5), 2), np.pad(a, 1).shape)

(4, 2)
(2, 4)
(2, 2, 2) (2, 2, 2)
(4, 2) (2, 4)
[[ 1 10]
 [ 2 20]
 [ 3 30]]
[[0, 1, 2, 3], [4, 5, 6], [7, 8, 9]]
[3 4 0 1 2] (4, 4)


## 16.8 Sorting, Searching, Unique, where and NaN Handling

In [ ]:
import numpy as np
names = np.array(["Asha", "Ben", "Chen", "Dana", "Eli"])
score = np.array([72, 95, 88, 95, 60])
order = np.argsort(score)
print(order, names[order])
desc = np.argsort(-score, kind="stable")
print(names[desc])
top3 = np.argpartition(-score, 3)[:3]
print(sorted(top3.tolist()), names[top3[np.argsort(-score[top3])]])
ranks = np.empty_like(order)
ranks[order] = np.arange(len(score))
print(ranks)
sorted_scores = np.sort(score)
print(np.searchsorted(sorted_scores, [60, 90, 100]))

[4 0 2 1 3] ['Eli' 'Asha' 'Chen' 'Ben' 'Dana']
['Ben' 'Dana' 'Chen' 'Asha' 'Eli']
[1, 2, 3] ['Dana' 'Ben' 'Chen']
[1 3 2 4 0]
[0 3 5]


In [ ]:
import numpy as np
city = np.array(["Pune", "Oslo", "Pune", "Lima", "Oslo", "Pune"])
vals, counts = np.unique(city, return_counts=True)
print(vals, counts)
vals, codes = np.unique(city, return_inverse=True)
print(codes)
print(np.bincount(codes))
spend = np.array([0.0, 4.5, 12.0, 25.0, 99.0, 7.0])
bins = np.array([5, 20])
idx = np.digitize(spend, bins)
print(idx)
label = np.select([spend < 5, spend < 20], ["low", "mid"], default="high")
print(label)
print(np.where(spend > 10, "big", "small"))
print(np.isin(city, ["Lima", "Oslo"]))

['Lima' 'Oslo' 'Pune'] [1 2 3]
[2 1 2 0 1 2]
[1 2 3]
[0 0 1 2 2 1]
['low' 'low' 'mid' 'high' 'high' 'mid']
['small' 'small' 'big' 'big' 'big' 'small']
[False  True False  True  True False]


In [ ]:
import numpy as np
x = np.array([4.0, np.nan, 6.0, 10.0, np.nan])
print(np.nan == np.nan, np.isnan(x))
print(x.mean(), np.nanmean(x), np.nanstd(x))
print(np.nanmax(x), np.nanargmax(x))
mask = np.isnan(x)
filled_mean = np.where(mask, np.nanmean(x), x)
print(filled_mean)
print(np.nan_to_num(x, nan=-1.0))
idx = np.arange(len(x))
interp = x.copy()
interp[mask] = np.interp(idx[mask], idx[~mask], x[~mask])
print(interp)
print(np.count_nonzero(mask), mask.mean())

False [False  True False False  True]
nan 6.666666666666667 2.494438257849294
10.0 3
[ 4.          6.66666667  6.         10.          6.66666667]
[ 4. -1.  6. 10. -1.]
[ 4.  5.  6. 10. 10.]
2 0.4


## 16.9 Linear Algebra with np.linalg

Linear algebra is the maths of tables of numbers, and it is the engine inside regression, PCA, recommender systems and neural networks (see the Mathematics document, Chapters 5 to 9, for the theory). NumPy hands the heavy work to optimised BLAS and LAPACK libraries.

In [ ]:
import numpy as np
A = np.array([[2.0, 1.0], [1.0, 3.0]])
b = np.array([7.0, 11.0])
x = np.linalg.solve(A, b)
print(x)
print(np.allclose(A @ x, b))
print(np.linalg.det(A), np.linalg.matrix_rank(A))
Ainv = np.linalg.inv(A)
print(np.round(A @ Ainv, 10))
B = np.array([[0.0, 1.0], [1.0, 0.0]])
print(A @ B)
print(A * B)
print(np.linalg.norm([3, 4]), np.trace(A))

[2. 3.]
True
5.000000000000001 2
[[ 1.  0.]
 [-0.  1.]]
[[1. 2.]
 [3. 1.]]
[[0. 1.]
 [1. 0.]]
5.0 5.0


## 16.10 Random Numbers: Generator and Seeds

Modern NumPy uses a **Generator**: an object that produces random numbers from a seeded stream. Create it once with `rng = np.random.default_rng(seed)` and call methods on it. The old global functions (`np.random.seed`, `np.random.rand`) still work but are legacy; the Generator is the recommended API because it is faster, has better statistical quality, and avoids hidden global state.

In [ ]:
import numpy as np
r1 = np.random.default_rng(42)
r2 = np.random.default_rng(42)
print(r1.random(3), r2.random(3))
print(np.array_equal(np.random.default_rng(1).random(5), np.random.default_rng(2).random(5)))
rng = np.random.default_rng(0)
rolls = rng.integers(1, 7, size=10)
print(rolls, rolls.min() >= 1, rolls.max() <= 6)
print(rng.choice(["red", "green", "blue"], size=8, p=[0.6, 0.3, 0.1]))
print(rng.choice(10, size=4, replace=False))
z = rng.normal(100, 15, size=100000)
print(round(float(z.mean()), 1), round(float(z.std()), 1))

[0.77395605 0.43887844 0.85859792] [0.77395605 0.43887844 0.85859792]
False
[6 4 4 2 2 1 1 1 2 5] True True
['blue' 'green' 'green' 'red' 'blue' 'green' 'red' 'green']
[0 6 3 7]
100.0 15.0


## 16.11 einsum: Index Notation for Array Algebra

`np.einsum` writes a calculation as a recipe of axis letters. Letters that appear in the inputs but not after the arrow `->` are **summed over**; letters that appear after the arrow stay. Once you can read it, many tangled `reshape`, `transpose` and `sum` chains become one line.

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
A = rng.integers(0, 5, (3, 4))
B = rng.integers(0, 5, (4, 2))
u = np.array([1, 2, 3])
v = np.array([4, 5, 6])
print(np.einsum("i,i->", u, v), u @ v)
print(np.array_equal(np.einsum("ij,jk->ik", A, B), A @ B))
print(np.array_equal(np.einsum("ij->i", A), A.sum(axis=1)))
print(np.einsum("i,j->ij", u, v))
batch = rng.integers(0, 3, (5, 2, 3))
other = rng.integers(0, 3, (5, 3, 4))
print(np.einsum("bij,bjk->bik", batch, other).shape)
P = rng.random((4, 3))
Q = rng.random((4, 3))
print(np.allclose(np.einsum("ij,ij->i", P, Q), (P * Q).sum(axis=1)))

32 32
True
True
[[ 4  5  6]
 [ 8 10 12]
 [12 15 18]]
(5, 2, 4)
True


## 16.12 Vectorisation vs Loops, with Timings

Vectorising means replacing a Python `for` loop with array operations. The loop is slow not because Python is careless but because for each element it must look up types, create a Python float object and call the interpreter. A NumPy operation does that bookkeeping once, then runs a tight compiled loop over raw numbers.

In [ ]:
import time
import numpy as np

def best(f, repeats=3):
    t = float("inf")
    for _ in range(repeats):
        start = time.perf_counter()
        f()
        t = min(t, time.perf_counter() - start)
    return t

rng = np.random.default_rng(0)
x = rng.random(1_000_000)
xl = x.tolist()

def loop_scale():
    out = []
    for v in xl:
        out.append(v * 2 + 1)
    return out

t_loop = best(loop_scale)
t_vec = best(lambda: x * 2 + 1)
print("same result:", np.allclose(loop_scale(), x * 2 + 1))
print("vectorised at least 10x faster:", t_loop / t_vec > 10)

t_sum_loop = best(lambda: sum(v * v for v in xl))
t_sum_vec = best(lambda: x @ x)
print("sum of squares equal:", np.isclose(sum(v * v for v in xl), x @ x))
print("at least 10x faster:", t_sum_loop / t_sum_vec > 10)

def grow():
    a = np.array([])
    for i in range(10000):
        a = np.append(a, i)
    return a

def preallocate():
    a = np.empty(10000)
    for i in range(10000):
        a[i] = i
    return a

print("growing is slower than preallocating:", best(grow) > best(preallocate))

same result: True
vectorised at least 10x faster: True
sum of squares equal: True
at least 10x faster: True
growing is slower than preallocating: True


## 16.13 Views, Copies and Memory Pitfalls

Because arrays share buffers, the question "does this change my original?" matters. The answer follows a simple rule.

In [ ]:
# This cell is meant to fail: read the error message, that is the lesson.
import numpy as np

def centre_bad(x):
    x -= x.mean()
    return x

def centre_good(x):
    return x - x.mean()

data = np.array([2.0, 4.0, 6.0])
out = centre_bad(data)
print(data, out, out is data)

data = np.array([2.0, 4.0, 6.0])
out = centre_good(data)
print(data, out)

ints = np.array([1, 2, 3])
try:
    ints /= 2
except Exception as e:
    print(type(e).__name__)
ints = ints / 2
print(ints, ints.dtype)

a = np.arange(6)
row = a[:3]
row *= 10
print(a)
print(np.shares_memory(a, a[:3]), np.shares_memory(a, a[[0, 1, 2]]))

[-2.  0.  2.] [-2.  0.  2.] True
[2. 4. 6.] [-2.  0.  2.]
UFuncTypeError
[0.5 1.  1.5] float64
[ 0 10 20  3  4  5]
True False


## 16.14 Recipes: How Do I ...?

Each recipe is a complete, runnable answer to a task you will meet again. Copy the pattern, change the names.

In [ ]:
import numpy as np
X = np.array([[170.0, 65.0, 30.0],
              [180.0, 80.0, 45.0],
              [160.0, 55.0, 22.0],
              [175.0, 90.0, 38.0]])
lo, hi = X.min(axis=0), X.max(axis=0)
minmax = (X - lo) / (hi - lo)
mu, sd = X.mean(axis=0), X.std(axis=0)
z = (X - mu) / sd
print(minmax.round(2))
print(z.mean(axis=0).round(6), z.std(axis=0).round(6))
const = np.array([[5.0, 1.0], [5.0, 2.0]])
rng_ = np.ptp(const, axis=0)
safe = np.where(rng_ == 0, 1.0, rng_)
print((const - const.min(axis=0)) / safe)

[[0.5  0.29 0.35]
 [1.   0.71 1.  ]
 [0.   0.   0.  ]
 [0.75 1.   0.7 ]]
[0. 0. 0.] [1. 1. 1.]
[[0. 0.]
 [0. 1.]]


In [ ]:
import numpy as np
labels = np.array(["cat", "dog", "bird", "dog", "cat"])
classes, codes = np.unique(labels, return_inverse=True)
onehot = np.eye(len(classes), dtype=int)[codes]
print(classes)
print(onehot)
back = classes[onehot.argmax(axis=1)]
print(back)
alt = np.zeros((len(codes), len(classes)), dtype=int)
alt[np.arange(len(codes)), codes] = 1
print(np.array_equal(alt, onehot))

['bird' 'cat' 'dog']
[[0 1 0]
 [0 0 1]
 [1 0 0]
 [0 0 1]
 [0 1 0]]
['cat' 'dog' 'bird' 'dog' 'cat']
True


In [ ]:
import numpy as np
pts = np.array([[0.0, 0.0], [3.0, 4.0], [6.0, 8.0], [0.0, 1.0]])
diff = pts[:, None, :] - pts[None, :, :]
D = np.sqrt((diff ** 2).sum(axis=-1))
print(D.round(2))
print(diff.shape)
sq = (pts ** 2).sum(axis=1)
D2 = np.sqrt(np.maximum(sq[:, None] + sq[None, :] - 2 * pts @ pts.T, 0))
print(np.allclose(D, D2))
from scipy.spatial.distance import cdist
print(np.allclose(D, cdist(pts, pts)))
nearest = np.where(np.eye(len(pts), dtype=bool), np.inf, D).argmin(axis=1)
print(nearest)

[[ 0.    5.   10.    1.  ]
 [ 5.    0.    5.    4.24]
 [10.    5.    0.    9.22]
 [ 1.    4.24  9.22  0.  ]]
(4, 4, 2)
True
True
[3 3 1 0]


In [ ]:
import numpy as np
x = np.array([1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0])
w = 3
c = np.cumsum(np.insert(x, 0, 0.0))
ma = (c[w:] - c[:-w]) / w
print(ma)
conv = np.convolve(x, np.ones(w) / w, mode="valid")
print(np.allclose(ma, conv))
from numpy.lib.stride_tricks import sliding_window_view
win = sliding_window_view(x, w)
print(win.shape, win.mean(axis=1))

[2. 3. 4. 5. 6. 7.]
True
(6, 3) [2. 3. 4. 5. 6. 7.]


In [ ]:
import numpy as np
scores = np.array([0.2, 0.9, 0.4, 0.95, 0.1, 0.7])
k = 3
idx = np.argpartition(scores, -k)[-k:]
top = idx[np.argsort(scores[idx])[::-1]]
print(top, scores[top])
M = np.array([[5, 1, 9, 3], [2, 8, 4, 6], [7, 7, 1, 0]])
order = np.argsort(-M, axis=1, kind="stable")[:, :2]
print(order)
print(np.take_along_axis(M, order, axis=1))
print(np.sort(M, axis=1)[:, ::-1][:, :2])

[3 1 5] [0.95 0.9  0.7 ]
[[2 0]
 [1 3]
 [0 1]]
[[9 5]
 [8 6]
 [7 7]]
[[9 5]
 [8 6]
 [7 7]]


In [ ]:
import numpy as np
rng = np.random.default_rng(0)
img = rng.integers(0, 256, size=(4, 6, 3), dtype=np.uint8)
print(img.shape, img.dtype, img.min(), img.max())
gray = img @ np.array([0.299, 0.587, 0.114])
print(gray.shape, gray.dtype)
gray8 = gray.round().astype(np.uint8)
flipped = img[:, ::-1]
crop = img[1:3, 2:5]
print(flipped.shape, crop.shape, np.shares_memory(img, crop))
bright = np.clip(img.astype(np.int16) + 100, 0, 255).astype(np.uint8)
wrong = img + np.uint8(100)
print((bright[0, 0] >= img[0, 0]).all(), (wrong[0, 0] >= img[0, 0]).all())
chw = np.moveaxis(img, -1, 0)
print(chw.shape, (img / 255.0).dtype)
mask = gray8 > 128
print(mask.sum(), img[mask].shape)

(4, 6, 3) uint8 1 248
(4, 6) float64
(4, 6, 3) (2, 3, 3) True
True False
(3, 4, 6) float64
12 (12, 3)


In [ ]:
import numpy as np
rng = np.random.default_rng(7)
for n in (100, 10_000, 1_000_000):
    pts = rng.random((n, 2))
    inside = (pts ** 2).sum(axis=1) <= 1.0
    print(n, round(4 * inside.mean(), 4))
rolls = rng.integers(1, 7, size=(100000, 2)).sum(axis=1)
print(round(float((rolls == 7).mean()), 3), round(6 / 36, 3))

100 3.08
10000 3.162
1000000 3.1437
0.166 0.167


In [ ]:
import numpy as np
def softmax_naive(z):
    e = np.exp(z)
    return e / e.sum(axis=-1, keepdims=True)

def softmax(z):
    z = z - z.max(axis=-1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=-1, keepdims=True)

z = np.array([1000.0, 1001.0, 1002.0])
with np.errstate(all="ignore"):
    print(softmax_naive(z))
print(softmax(z).round(4))
print(np.allclose(softmax(np.array([2.0, 1.0, 0.1])), softmax(np.array([102.0, 101.0, 100.1]))))
batch = np.array([[2.0, 1.0, 0.1], [0.0, 0.0, 0.0]])
print(softmax(batch).round(3), softmax(batch).sum(axis=1))
def log_softmax(z):
    z = z - z.max(axis=-1, keepdims=True)
    return z - np.log(np.exp(z).sum(axis=-1, keepdims=True))
print(log_softmax(z).round(4))

[nan nan nan]
[0.09   0.2447 0.6652]
True
[[0.659 0.242 0.099]
 [0.333 0.333 0.333]] [1. 1.]
[-2.4076 -1.4076 -0.4076]


In [ ]:
import numpy as np
rng = np.random.default_rng(42)
X = np.arange(20).reshape(10, 2)
y = np.array([0, 1] * 5)
perm = rng.permutation(len(X))
n_test = int(len(X) * 0.3)
test_idx, train_idx = perm[:n_test], perm[n_test:]
X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
print(X_train.shape, X_test.shape)
print(sorted(train_idx.tolist() + test_idx.tolist()) == list(range(10)))
print(y_test, y_train.mean())
def stratified_split(y, test_frac, rng):
    test = []
    for cls in np.unique(y):
        members = rng.permutation(np.flatnonzero(y == cls))
        test.append(members[: round(len(members) * test_frac)])
    test = np.concatenate(test)
    train = np.setdiff1d(np.arange(len(y)), test)
    return train, test
tr, te = stratified_split(np.array([0] * 8 + [1] * 2), 0.5, np.random.default_rng(0))
print(len(tr), len(te), np.bincount(np.array([0] * 8 + [1] * 2)[te]))

(7, 2) (3, 2)
True
[1 0 0] 0.5714285714285714
5 5 [4 1]


In [ ]:
import numpy as np
store = np.array([0, 2, 1, 0, 2, 2, 1])
amount = np.array([10.0, 5.0, 7.5, 2.5, 1.0, 4.0, 2.5])
print(np.bincount(store, weights=amount))
print(np.bincount(store))
totals = np.zeros(3)
np.add.at(totals, store, amount)
print(totals)
ages = np.array([3, 17, 25, 31, 44, 52, 67, 71])
counts, edges = np.histogram(ages, bins=[0, 18, 40, 65, 120])
print(counts, edges)
print(np.digitize(ages, [18, 40, 65]))

[12.5 10.  10. ]
[2 2 3]
[12.5 10.  10. ]
[2 2 2 2] [  0  18  40  65 120]
[0 0 1 1 2 2 3 3]


## 16.15 Performance and Pitfalls

In [ ]:
import numpy as np
a = np.array([1, 2, 3])
a[0] = 2.9
print(a)
f = np.float32(16777216)
print(f + np.float32(1) == f)
y = np.array([1.0, 2.0, 3.0])
pred = np.array([[1.0], [2.0], [3.0]])
print((y - pred).shape, ((y - pred) ** 2).mean())
print(((y - pred.ravel()) ** 2).mean())
b = a
b[1] = 99
print(a)
data = np.array([3, 1, 2])
result = data.sort()
print(result, data)
v = np.array([1.0, np.nan, 3.0])
print(v.argmax(), np.nanargmax(v))
M = np.arange(6).reshape(2, 3)
print(sum(M), M.sum())

[2 2 3]
True
(3, 3) 1.3333333333333333
0.0
[ 2 99  3]
None [1 2 3]
1 2
[3 5 7] 15


## 16.16 Cross-Library Integration

NumPy is the common language of the scientific Python world. Knowing where memory is shared and where it is copied lets you pass data between libraries without surprises. This section was run with pandas 3.0.6, SciPy 1.18.1, scikit-learn 1.9.1 and matplotlib 3.11.2.

In [ ]:
# This cell is meant to fail: read the error message, that is the lesson.
import numpy as np
import pandas as pd
df = pd.DataFrame({"price": [10.0, 12.5, 9.0], "qty": [3, 1, 4], "name": ["a", "b", "c"]})
print({k: str(v) for k, v in df.dtypes.items()})
nums = df[["price", "qty"]].to_numpy()
print(nums.dtype, nums.shape)
col = df["price"].to_numpy()
print(col.flags.writeable)
try:
    col[0] = 99.0
except ValueError as e:
    print("ValueError:", e)
safe = df["price"].to_numpy().copy()
safe[0] = 99.0
print(df["price"].iloc[0], safe[0])
print(np.log(df["price"]).round(3).tolist())
df["total"] = df["price"] * df["qty"]
print(df["total"].to_numpy(), np.sum(df["total"]))
back = pd.DataFrame(np.arange(6).reshape(3, 2), columns=["x", "y"])
print(back.dtypes.to_dict())

{'price': 'float64', 'qty': 'int64', 'name': 'str'}
float64 (3, 2)
False
ValueError: assignment destination is read-only
10.0 99.0
[2.303, 2.526, 2.197]
[30.  12.5 36. ] 78.5
{'x': dtype('int64'), 'y': dtype('int64')}


In [ ]:
# This cell is meant to fail: read the error message, that is the lesson.
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
rng = np.random.default_rng(3)
X = rng.normal(10, 2, size=(100, 3))
y = X @ np.array([1.0, -2.0, 0.5]) + 4.0 + rng.normal(0, 0.1, 100)
sc = StandardScaler().fit(X)
print(np.allclose(sc.transform(X), (X - X.mean(axis=0)) / X.std(axis=0)))
model = LinearRegression().fit(X, y)
print(model.coef_.round(2), round(float(model.intercept_), 2))
A = np.column_stack([X, np.ones(len(X))])
coef = np.linalg.lstsq(A, y, rcond=None)[0]
print(coef.round(2))
try:
    LinearRegression().fit(np.arange(5.0), np.arange(5.0))
except ValueError:
    print("ValueError: X must be 2D")
print(LinearRegression().fit(np.arange(5.0).reshape(-1, 1), np.arange(5.0) * 2).coef_)

True
[ 1.   -1.99  0.5 ] 3.85
[ 1.   -1.99  0.5   3.85]
ValueError: X must be 2D
[2.]


## 16.17 How to Solve Any NumPy Problem: a Playbook

In [ ]:
import numpy as np
cust = np.array([0, 1, 0, 2, 1, 0, 2, 1])
price = np.array([20.0, 35.0, 15.0, 50.0, 10.0, 25.0, 40.0, 30.0])
qty = np.array([1, 2, 3, 1, 5, 2, 1, 1])
rating = np.array([4.0, np.nan, 5.0, 3.0, 4.0, np.nan, 2.0, 5.0])
revenue = price * qty

size = np.where(revenue >= 60, "large", "small")
print("1", size)

table = np.zeros((3, 2))
for c in range(3):
    table[c] = [revenue[cust == c].sum(), qty[cust == c].sum()]
by_cust = np.bincount(cust, weights=revenue)
print("2", by_cust, table[:, 0])

P = np.column_stack([price, qty]).astype(float)
D = np.abs(P[:, None, 0] - P[None, :, 0])
print("3", D.shape, D[0, :3])

print("4", np.bincount(cust), np.unique(cust, return_counts=True)[1])

running = np.cumsum(revenue)
print("5", running, np.cumsum(revenue)[-1] == revenue.sum())

ok = ~np.isnan(rating)
print("6", rating[ok], round(float(rating[ok].mean()), 2), round(float(np.nanmean(rating)), 2))

top2 = np.argsort(-revenue, kind="stable")[:2]
print("7", top2, revenue[top2])

A = np.column_stack([price, np.ones(len(price))])
coef = np.linalg.lstsq(A, qty, rcond=None)[0]
print("8", coef.round(3))

1 ['small' 'large' 'small' 'small' 'small' 'small' 'small' 'small']
2 [115. 150.  90.] [115. 150.  90.]
3 (8, 8) [ 0. 15.  5.]
4 [3 3 2] [3 3 2]
5 [ 20.  90. 135. 185. 235. 285. 325. 355.] True
6 [4. 5. 3. 4. 2. 5.] 3.83 3.83
7 [1 3] [70. 50.]
8 [-0.076  4.143]


In [ ]:
import numpy as np
rng = np.random.default_rng(11)
P = rng.random((40, 3))
D = np.sqrt(((P[:, None] - P[None]) ** 2).sum(-1))
ref = np.zeros((40, 40))
for i in range(40):
    for j in range(40):
        ref[i, j] = np.sqrt(sum((P[i, k] - P[j, k]) ** 2 for k in range(3)))
np.testing.assert_allclose(D, ref, rtol=1e-12)
print("matches loop:", True)
print("symmetric:", np.allclose(D, D.T), "zero diagonal:", np.allclose(np.diag(D), 0))
print("triangle inequality holds:", bool((D[0, 1] + D[1, 2] >= D[0, 2] - 1e-12)))

matches loop: True
symmetric: True zero diagonal: True
triangle inequality holds: True


## 16.18 Going Deeper: Internals, Memory, Speed and Interview Questions

In [ ]:
import numpy as np
print(np.add.reduce([1, 2, 3]), np.add.accumulate([1, 2, 3]))
print(np.multiply.outer([1, 2], [3, 4]))
print(np.maximum.accumulate([1, 3, 2, 5, 4]))
a = np.zeros(3)
np.add.at(a, [0, 0, 2], 1.0)
print(a)
print(np.add.types[:4])

6 [1 3 6]
[[3 4]
 [6 8]]
[1 3 3 5 5]
[2. 0. 1.]
['??->?', 'bb->b', 'BB->B', 'hh->h']


In [ ]:
# This cell is meant to fail: read the error message, that is the lesson.
import numpy as np
a = np.array([200], dtype=np.uint8)
print((a + 100), (a + 100).dtype)
print((np.array([1.0], dtype=np.float32) + 3.0).dtype)
try:
    np.array([1], dtype=np.uint8) + 300
except OverflowError as e:
    print("OverflowError:", e)
print(np.result_type(np.uint64, np.int64), np.result_type(np.int32, np.float32))

[44] uint8
float32
OverflowError: Python integer 300 out of bounds for uint8
float64 float64


## Chapter recap

## Practice set

Try each problem before you read the answer. Problems are graded from easy to expert. Code shown in the problems is plain text; run it yourself to check.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/python/ch16.html